In [1]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import requests
import pandas as pd
df=pd.DataFrame()
from datetime import datetime


In [3]:
all_records = []
start_year = datetime.now().year -5
end_year = datetime.now().year

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

In [4]:
for year in range(start_year,end_year + 1):
    for month in range(1, 13):
        start_date = f"{year}-{month:02d}-01"
        if month == 12:
            end_date = f"{year+1}-01-01"
        else:
            end_date = f"{year}-{month+1:02d}-01"

        params = {
            "format": "geojson",
            "starttime": start_date,
            "endtime": end_date,
            "minmagnitude": 3
        }

        response = requests.get(url, params=params,timeout=30)
        if response.status_code != 200:
            print(f" Failed for {start_date}: {response.text[:200]}")
            continue

        try:
            data = response.json()
        except Exception as e:
            print(f" JSON error for {start_date}: {e}")
            continue

        for f in data["features"]:
            p = f["properties"]
            g = f["geometry"]["coordinates"]
            all_records.append({
                "id": f.get("id"),
                "time": pd.to_datetime(p.get("time"), unit="ms"),
                "updated": pd.to_datetime(p.get("updated"), unit="ms"),
                "latitude": g[1] if g else None,
                "longitude": g[0] if g else None,
                "depth_km": g[2] if g else None,
                "mag": p.get("mag"),
                "magType": p.get("magType"),
                "place":p.get("place"),
                "status":p.get("status"),
                "tsunami":p.get("tsunami"),
                "alert":p.get("alert"),
                "felt":p.get("felt"),
                "cdi":p.get("cdi"),
                "sig":p.get("sig"),
                "net":p.get("net"),
                "ids":p.get("ids"),
                "sources":p.get("sources"),
                "types":p.get("types"),
                "nst":p.get("nst"),
                "dmin":p.get("dmin"),
                "rms":p.get("rms"),
                "gap":p.get("gap"),
                "type":p.get("type"),
                "magSource":p.get("magSource"),
                "locationSource":p.get("locationSource"),
                "magNst":p.get("magNst"),
                "magError":p.get("magError"),
                "depthError":p.get("depthError")

            })

df=pd.DataFrame(all_records)

print("Rows:",df.shape[0])
print("Columns:",df.shape[1])
print(df.head())

Rows: 119347
Columns: 29
           id                    time                 updated  latitude  \
0  us6000ddi8 2021-01-31 23:20:49.923 2021-04-16 19:02:44.040  -31.7493   
1  us6000dev6 2021-01-31 23:08:17.161 2021-04-16 19:03:47.040  -15.4902   
2  us6000dev5 2021-01-31 22:54:19.760 2021-04-16 19:03:47.040   19.7529   
3  us6000ddhs 2021-01-31 22:06:00.832 2021-04-16 19:02:43.040   28.1524   
4  us6000dev4 2021-01-31 21:51:14.016 2021-04-16 19:03:46.040   71.3212   

   longitude  depth_km  mag magType  \
0   -68.9337     17.27  4.7     mwr   
1  -177.2052    426.71  4.1      mb   
2   121.3159     46.73  4.7      mb   
3    57.2570     10.00  4.9      mb   
4    -3.7578     10.00  4.0      mb   

                                               place    status  ...  nst  \
0        29 km SW of Villa Basilio Nievas, Argentina  reviewed  ...  NaN   
1                                        Fiji region  reviewed  ...  NaN   
2                    103 km SW of Basco, Philippines  reviewe

In [5]:
print(df)

                   id                    time                 updated  \
0          us6000ddi8 2021-01-31 23:20:49.923 2021-04-16 19:02:44.040   
1          us6000dev6 2021-01-31 23:08:17.161 2021-04-16 19:03:47.040   
2          us6000dev5 2021-01-31 22:54:19.760 2021-04-16 19:03:47.040   
3          us6000ddhs 2021-01-31 22:06:00.832 2021-04-16 19:02:43.040   
4          us6000dev4 2021-01-31 21:51:14.016 2021-04-16 19:03:46.040   
...               ...                     ...                     ...   
119342     us7000tdbz 2026-09-01 04:50:45.457 2026-09-01 20:23:31.864   
119343  aka2026rhblzv 2026-09-01 03:04:24.961 2026-09-01 16:13:24.871   
119344     pr71530493 2026-09-01 03:03:51.600 2026-09-01 09:34:39.270   
119345     us7000td94 2026-09-01 02:22:48.557 2026-09-01 16:08:36.998   
119346     us7000td8u 2026-09-01 01:29:10.774 2026-09-01 01:44:48.040   

        latitude  longitude  depth_km   mag magType  \
0       -31.7493   -68.9337    17.270  4.70     mwr   
1       -15.4

In [6]:
##CLEAN TEXT FIELDS##

import pandas as pd
import numpy as np
import re

In [7]:
df["country"] = df["place"].str.extract(r",\s*([^,]+)$",expand=False)
df["country"] = df["country"].fillna("Unknown")
print(df.country)

0                      Argentina
1                        Unknown
2                    Philippines
3                           Iran
4         Svalbard and Jan Mayen
                   ...          
119342                    Panama
119343                    Alaska
119344        Dominican Republic
119345                    Alaska
119346                     Tonga
Name: country, Length: 119347, dtype: str


In [8]:
if "alert" in df.columns:
 df["alert"] = df["alert"].astype("string").str.strip().str.lower()
 print(df.alert)

0         <NA>
1         <NA>
2         <NA>
3         <NA>
4         <NA>
          ... 
119342    <NA>
119343    <NA>
119344    <NA>
119345    <NA>
119346    <NA>
Name: alert, Length: 119347, dtype: string


In [9]:
string_columns = [
  "magType",
  "status",
  "type",
  "net",
  "sources",
  "types"
]
for col in string_columns:
 if col in df.columns:
  df[col] = (df[col].astype("string") .str.strip().str.lower())

In [10]:
df.columns

Index(['id', 'time', 'updated', 'latitude', 'longitude', 'depth_km', 'mag',
       'magType', 'place', 'status', 'tsunami', 'alert', 'felt', 'cdi', 'sig',
       'net', 'ids', 'sources', 'types', 'nst', 'dmin', 'rms', 'gap', 'type',
       'magSource', 'locationSource', 'magNst', 'magError', 'depthError',
       'country'],
      dtype='str')

In [11]:
##CLEAN NUMERIC FIELDS##
numeric_columns =[
    "mag",
    "depth_km",
    "nst",
    "dmin",
    "rms",
    "gap",
    "magError",
    "depthError",
    "magNst",
    "sig"
]
for col in numeric_columns:
  if col in df.columns:
   df[col] = pd.to_numeric(df[col], errors="coerce")

for col in numeric_columns:
    if col in df.columns:
     df[col] = df[col].fillna(df[col].median())
print(df[numeric_columns].head())
print(df[numeric_columns].isnull().sum())

zero_fill_cols = ["magError", "depthError", "magNst", "sig"]
for col in zero_fill_cols:
    if col in df.columns:
     df[col] = df[col].fillna(0)
        
print(df[numeric_columns].dtypes)
print(df[numeric_columns].isnull().sum())

   mag  depth_km   nst   dmin   rms    gap  magError  depthError  magNst  sig
0  4.7     17.27  32.0  0.294  0.82   42.0       NaN         NaN     NaN  344
1  4.1    426.71  32.0  1.471  0.29   64.0       NaN         NaN     NaN  259
2  4.7     46.73  32.0  3.057  0.69  106.0       NaN         NaN     NaN  340
3  4.9     10.00  32.0  3.330  0.61   71.0       NaN         NaN     NaN  369
4  4.0     10.00  32.0  6.023  0.50   65.0       NaN         NaN     NaN  246
mag                0
depth_km           0
nst                0
dmin               0
rms                0
gap                0
magError      119347
depthError    119347
magNst        119347
sig                0
dtype: int64
mag           float64
depth_km      float64
nst           float64
dmin          float64
rms           float64
gap           float64
magError      float64
depthError    float64
magNst        float64
sig             int64
dtype: object
mag           0
depth_km      0
nst           0
dmin          0
rms        

In [12]:
##DERIVED COLUMNS##
df["time"] = pd.to_datetime(df["time"], errors="coerce")
df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month
df["day"] = df["time"].dt.day
df["day_of_week"] = df["time"].dt.day_name()

In [13]:
df["Shallow"] = np.where(df["depth_km"] < 50, "Yes", "No")
df["magnitude_category"] = np.select(
    [df["mag"] < 6.0, (df["mag"] >= 6.0) & (df["mag"] < 7.0), df["mag"] >= 7.0],
    ["normal", "strong", "destructive"],
    default="unknown"
)
print(df[["Shallow", "magnitude_category"]].value_counts())

Shallow  magnitude_category
Yes      normal                80919
No       normal                37654
Yes      strong                  504
No       strong                  184
Yes      destructive              60
No       destructive              26
Name: count, dtype: int64


In [14]:
!pip install mysql-connector-python


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import mysql.connector
conn_mysql=mysql.connector.connect(
    host="localhost",
    user="root",
    password="abiramisql121089"
)
cursor_mysql=conn_mysql.cursor()
print("Mysql connection established")

Mysql connection established


In [16]:
cursor_mysql.execute("CREATE DATABASE IF NOT EXISTS earthquake_db;")
cursor_mysql.execute("USE earthquake_db;")
cursor_mysql.execute("""
CREATE TABLE IF NOT EXISTS earthquake_data (
    id VARCHAR(255) PRIMARY KEY,
    time DATETIME,
    updated DATETIME,
    latitude FLOAT,
    longitude FLOAT,
    depth_km FLOAT,
    mag FLOAT,
    magType VARCHAR(20),
    place VARCHAR(255),
    status VARCHAR(50),
    tsunami INT,
    alert VARCHAR(20),
    felt INT,
    cdi FLOAT,
    sig INT,
    net VARCHAR(20),
    ids TEXT,
    sources TEXT,
    types TEXT,
    nst INT,
    dmin FLOAT,
    rms FLOAT,
    gap FLOAT,
    type VARCHAR(50),
    magSource VARCHAR(50),
    locationSource VARCHAR(50),
    magNst INT,
    magError FLOAT,
    depthError FLOAT,
    country VARCHAR(100),
    year INT,
    month INT,
    day INT,
    day_of_week VARCHAR(20),
    Shallow VARCHAR(20),
    magnitude_category VARCHAR(20)
);
""")
conn_mysql.commit()
print("Database and table created successfully!")

Database and table created successfully!


In [17]:
print(len(df.columns))
print(df.columns.tolist())
print(df.shape)

36
['id', 'time', 'updated', 'latitude', 'longitude', 'depth_km', 'mag', 'magType', 'place', 'status', 'tsunami', 'alert', 'felt', 'cdi', 'sig', 'net', 'ids', 'sources', 'types', 'nst', 'dmin', 'rms', 'gap', 'type', 'magSource', 'locationSource', 'magNst', 'magError', 'depthError', 'country', 'year', 'month', 'day', 'day_of_week', 'Shallow', 'magnitude_category']
(119347, 36)


In [18]:
print("number of rows:",len(df))
print(df.head())
print(df.columns.tolist())

number of rows: 119347
           id                    time                 updated  latitude  \
0  us6000ddi8 2021-01-31 23:20:49.923 2021-04-16 19:02:44.040  -31.7493   
1  us6000dev6 2021-01-31 23:08:17.161 2021-04-16 19:03:47.040  -15.4902   
2  us6000dev5 2021-01-31 22:54:19.760 2021-04-16 19:03:47.040   19.7529   
3  us6000ddhs 2021-01-31 22:06:00.832 2021-04-16 19:02:43.040   28.1524   
4  us6000dev4 2021-01-31 21:51:14.016 2021-04-16 19:03:46.040   71.3212   

   longitude  depth_km  mag magType  \
0   -68.9337     17.27  4.7     mwr   
1  -177.2052    426.71  4.1      mb   
2   121.3159     46.73  4.7      mb   
3    57.2570     10.00  4.9      mb   
4    -3.7578     10.00  4.0      mb   

                                               place    status  ...  magNst  \
0        29 km SW of Villa Basilio Nievas, Argentina  reviewed  ...     0.0   
1                                        Fiji region  reviewed  ...     0.0   
2                    103 km SW of Basco, Philippines  

In [19]:
columns = [
    'id', 'time', 'updated', 'latitude', 'longitude',
    'depth_km', 'mag', 'magType', 'place', 'status',
    'tsunami', 'alert', 'felt', 'cdi', 'sig', 'net',
    'ids', 'sources', 'types', 'nst', 'dmin', 'rms',
    'gap', 'type', 'magSource', 'locationSource',
    'magNst', 'magError', 'depthError',
    'country', 'year', 'month', 'day', 'day_of_week',
    'Shallow', 'magnitude_category'
]

data = [
    tuple(row)
    for row in df[columns].itertuples(index=False, name=None)
]
print("Rows:", len(data), "| Values per row:", len(data[0]))

Rows: 119347 | Values per row: 36


In [20]:
insert_query = """
INSERT INTO earthquake_data (
    id, time, updated, latitude, longitude, depth_km, mag, magType,
    place, status, tsunami, alert, felt, cdi, sig, net, ids, sources,
    types, nst, dmin, rms, gap, type, magSource, locationSource,
    magNst, magError, depthError, country, year, month, day, day_of_week,
    Shallow, magnitude_category
) VALUES (
    %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s
)
"""
# convert NaN/NaT to None so MySQL accepts them
clean_data = [
    tuple(None if (pd.isna(v) if not isinstance(v, str) else False) else v for v in row)
    for row in data
]

cursor_mysql.executemany(insert_query, clean_data)
conn_mysql.commit()
print(f"Inserted {cursor_mysql.rowcount} rows")


Inserted 119347 rows


In [75]:
##Magnitude & Depth
##1.Top 10 strongest earthquakes (mag).

from tabulate import tabulate
query1 = """select id,country, mag from earthquake_data order by mag desc limit 10;"""
cursor_mysql.execute(query1)
result1 = cursor_mysql.fetchall()
headers = ["id","country","mag"]
print(tabulate(result1, headers=headers, tablefmt="grid"))

+--------------+-----------------------------------+-------+
| id           | country                           |   mag |
+==============+===================================+=======+
| us6000qw60   | Russia Earthquake                 |   8.8 |
+--------------+-----------------------------------+-------+
| ak0219neiszm | Alaska Earthquake                 |   8.2 |
+--------------+-----------------------------------+-------+
| us7000dflf   | New Zealand Earthquake            |   8.1 |
+--------------+-----------------------------------+-------+
| us6000f53e   | Unknown                           |   8.1 |
+--------------+-----------------------------------+-------+
| us7000qx2g   | Russia                            |   7.8 |
+--------------+-----------------------------------+-------+
| us6000jllz   | Kahramanmaras earthquake sequence |   7.8 |
+--------------+-----------------------------------+-------+
| us6000tkt2   | Indonesia                         |   7.8 |
+--------------+--------

In [22]:
##2.Top 10 deepest earthquakes (depth_km).
from tabulate import tabulate
query2 = """select id,depth_km from earthquake_data order by depth_km desc limit 10;"""
cursor_mysql.execute("USE earthquake_db")
cursor_mysql.execute(query2)
result2 = cursor_mysql.fetchall()
headers = ["id","depth_km"]
print(tabulate(result2, headers=headers, tablefmt="grid"))

+------------+------------+
| id         |   depth_km |
+============+============+
| us6000sp3p |    683.578 |
+------------+------------+
| us6000k2db |    681.238 |
+------------+------------+
| us7000kxdn |    675.265 |
+------------+------------+
| us6000mivr |    671.043 |
+------------+------------+
| us6000rk66 |    669.556 |
+------------+------------+
| us6000f2w3 |    669.46  |
+------------+------------+
| us7000q1jk |    667.237 |
+------------+------------+
| us7000s0st |    667.197 |
+------------+------------+
| us6000ta9m |    665.993 |
+------------+------------+
| us6000tknp |    665.326 |
+------------+------------+


In [23]:
##3.Shallow earthquakes < 50 km and mag > 7.5.
from tabulate import tabulate
query3 = """select id, depth_km, mag from earthquake_data where depth_km < 50 and mag > 7.5
order by mag desc;"""
cursor_mysql.execute("USE earthquake_db")
cursor_mysql.execute(query3)
result3 = cursor_mysql.fetchall()
headers = ["id", "depth_km", "mag"]
print(tabulate(result3, headers=headers, tablefmt="grid"))

+--------------+------------+-------+
| id           |   depth_km |   mag |
+==============+============+=======+
| us6000qw60   |     35     |   8.8 |
+--------------+------------+-------+
| ak0219neiszm |     35     |   8.2 |
+--------------+------------+-------+
| us6000f53e   |     22.79  |   8.1 |
+--------------+------------+-------+
| us7000dflf   |     28.93  |   8.1 |
+--------------+------------+-------+
| us6000jllz   |     10     |   7.8 |
+--------------+------------+-------+
| us6000tkt2   |     10     |   7.8 |
+--------------+------------+-------+
| us7000qx2g   |     27     |   7.8 |
+--------------+------------+-------+
| us6000dg77   |     10     |   7.7 |
+--------------+------------+-------+
| us6000kd0n   |     18.053 |   7.7 |
+--------------+------------+-------+
| us7000pn9s   |     10     |   7.7 |
+--------------+------------+-------+
| us6000rgf4   |      5.639 |   7.6 |
+--------------+------------+-------+
| us6000rtdt   |     40.72  |   7.6 |
+-----------

In [24]:
 ## 5.Average magnitude per magnitude type (magType).
from tabulate import tabulate
query5="""select magType,avg(mag) AS average_magnitude from earthquake_data 
group by magType order by average_magnitude desc;"""
cursor_mysql.execute("USE earthquake_db")
cursor_mysql.execute(query5)
result5 = cursor_mysql.fetchall()
headers = ["Mag_type","Avg_mag"]
print(tabulate(result5, headers=headers, tablefmt="grid"))

+------------+-----------+
| Mag_type   |   Avg_mag |
+============+===========+
| mwc        |   6.15    |
+------------+-----------+
| ms_20      |   5.8     |
+------------+-----------+
| mwb        |   5.7875  |
+------------+-----------+
| mww        |   5.36645 |
+------------+-----------+
| mwp        |   5.25    |
+------------+-----------+
| ms_vx      |   4.65    |
+------------+-----------+
| mb         |   4.42349 |
+------------+-----------+
| mwr        |   4.32425 |
+------------+-----------+
| mh         |   4.1     |
+------------+-----------+
| mw         |   3.93812 |
+------------+-----------+
| mlr        |   3.48    |
+------------+-----------+
| mlv        |   3.46929 |
+------------+-----------+
| md         |   3.43841 |
+------------+-----------+
| ml(texnet) |   3.36744 |
+------------+-----------+
| mlg        |   3.3     |
+------------+-----------+
| ml         |   3.27541 |
+------------+-----------+
| mb_lg      |   3.20315 |
+------------+-----------+


In [25]:
#Time Analysis
##6.Year with most earthquakes
from tabulate import tabulate
query6 = """select year(time) as year, count(*) as earthquake_count from earthquake_data
group by year(time)order by earthquake_count desc limit 1;"""
cursor_mysql.execute("USE earthquake_db")
cursor_mysql.execute(query6)
result6 = cursor_mysql.fetchall()
print(tabulate(result6, headers=["Year","Earthquake Count"],tablefmt="grid"))


+--------+--------------------+
|   Year |   Earthquake Count |
+========+====================+
|   2025 |              22819 |
+--------+--------------------+


In [26]:
##7. Month with highest number of earthquakes.
from tabulate import tabulate
query7 = """select month(time) as month_number,monthname(time) as month,count(*) AS earthquake_count
from earthquake_data group by month(time),monthname(time)
order by earthquake_count desc limit 1;"""
cursor_mysql.execute("USE earthquake_db")
cursor_mysql.execute(query7)
result7 = cursor_mysql.fetchall()
headers = ["month_number", "month", "earthquake_count"]
print(tabulate(result7, headers=headers, tablefmt="grid"))

+----------------+---------+--------------------+
|   month_number | month   |   earthquake_count |
+================+=========+====================+
|              7 | July    |              11791 |
+----------------+---------+--------------------+


In [27]:
##8.Day of week with most earthquakes.
from tabulate import tabulate
query8 = """select dayname(time) as dayofweek,count(*) as earthquake_count from earthquake_data
group by dayofweek(time),dayname(time)
order by earthquake_count desc limit 1;"""
cursor_mysql.execute("USE earthquake_db")
cursor_mysql.execute(query8)
result8 = cursor_mysql.fetchall()
headers =["day_of_week","earthquake_count"]
print(tabulate(result8, headers=headers, tablefmt="grid"))

+---------------+--------------------+
| day_of_week   |   earthquake_count |
+===============+====================+
| Friday        |              17252 |
+---------------+--------------------+


In [28]:
##9.Count of earthquakes per hour of day.
from tabulate import tabulate
query9="""select hour(time) as hour_of_day,count(*) as earthquake_count from earthquake_data
group by hour(time) order by hour_of_day;"""
cursor_mysql.execute("USE earthquake_db")
cursor_mysql.execute(query9)
result9 = cursor_mysql.fetchall()
headers =["hour_of_day","earthquake_count"]
print(tabulate(result9, headers=headers, tablefmt="grid"))

+---------------+--------------------+
|   hour_of_day |   earthquake_count |
+===============+====================+
|             0 |               4926 |
+---------------+--------------------+
|             1 |               5237 |
+---------------+--------------------+
|             2 |               5074 |
+---------------+--------------------+
|             3 |               5272 |
+---------------+--------------------+
|             4 |               5217 |
+---------------+--------------------+
|             5 |               4906 |
+---------------+--------------------+
|             6 |               4823 |
+---------------+--------------------+
|             7 |               4967 |
+---------------+--------------------+
|             8 |               4986 |
+---------------+--------------------+
|             9 |               4918 |
+---------------+--------------------+
|            10 |               4810 |
+---------------+--------------------+
|            11 |        

In [29]:
##10.Most active reporting network (net).
from tabulate import tabulate
query10 = """select net,count(*) as earthquake_count from earthquake_data 
group by net order by earthquake_count desc limit 1;"""
cursor_mysql.execute("USE earthquake_db")
cursor_mysql.execute(query10)
result10 = cursor_mysql.fetchall()
headers =["network","earthquake_count"]
print(tabulate(result10, headers=headers, tablefmt="grid"))

+-----------+--------------------+
| network   |   earthquake_count |
+===========+====================+
| us        |             104330 |
+-----------+--------------------+


In [30]:
##Casualties & Economic Loss

##11.Top 5 places with highest casualties.

from tabulate import tabulate
query11 = """select place, sum(felt) as total_casualties from earthquake_data 
group by place order by total_casualties desc limit 5;"""
cursor_mysql.execute("USE earthquake_db")
cursor_mysql.execute(query11)
result11= cursor_mysql.fetchall()
headers =["places","Highest_casualities"]
print(tabulate(result11, headers=headers, tablefmt="grid"))

+---------------------------------------+-----------------------+
| places                                |   Highest_casualities |
+=======================================+=======================+
| 2024 Tewksbury, New Jersey Earthquake |                184673 |
+---------------------------------------+-----------------------+
| 21 km SE of Greenback, Tennessee      |                 45935 |
+---------------------------------------+-----------------------+
| 5 km S of Julian, CA                  |                 43198 |
+---------------------------------------+-----------------------+
| 9 km SE of York Harbor, Maine         |                 42440 |
+---------------------------------------+-----------------------+
| 1 km SE of Boulder Creek, CA          |                 33078 |
+---------------------------------------+-----------------------+


In [31]:
##Event Type & Quality Metrics
##14.Count of reviewed vs automatic earthquakes (status).

from tabulate import tabulate
query14="""select status, Count(*) as earthquake_count from earthquake_data 
group by status order by earthquake_count DESC;"""
cursor_mysql.execute("USE earthquake_db")
cursor_mysql.execute(query14)
result14= cursor_mysql.fetchall()
headers =["status","earthquake_count"]
print(tabulate(result14, headers=headers, tablefmt="grid"))

+-----------+--------------------+
| status    |   earthquake_count |
+===========+====================+
| reviewed  |             119183 |
+-----------+--------------------+
| automatic |                164 |
+-----------+--------------------+


In [32]:
##15.Count by earthquake type (type).

from tabulate import tabulate
query15 = """select type,count(*) as earthquake_count from earthquake_data group by type 
order by earthquake_count desc;"""
cursor_mysql.execute("USE earthquake_db")
cursor_mysql.execute(query15)
result15 = cursor_mysql.fetchall()
headers = ["earthquake_type", "earthquake_count"]
print(tabulate(result15, headers=headers, tablefmt="grid"))

+------------------------+--------------------+
| earthquake_type        |   earthquake_count |
+========================+====================+
| earthquake             |             118449 |
+------------------------+--------------------+
| mining explosion       |                851 |
+------------------------+--------------------+
| ice quake              |                 13 |
+------------------------+--------------------+
| volcanic eruption      |                 12 |
+------------------------+--------------------+
| landslide              |                  6 |
+------------------------+--------------------+
| other event            |                  5 |
+------------------------+--------------------+
| experimental explosion |                  5 |
+------------------------+--------------------+
| mine collapse          |                  3 |
+------------------------+--------------------+
| quarry blast           |                  2 |
+------------------------+--------------

In [33]:
##16.Number of earthquakes by datatype (types).
query16 = """select types,count(*) as earthquake_count from earthquake_data group by types
order by earthquake_count desc;"""
cursor_mysql.execute("USE earthquake_db")
cursor_mysql.execute(query16)
result16 = cursor_mysql.fetchall()
print("datatype of earthquake:",result16)

datatype of earthquake: [(',origin,phase-data,', 90753), (',dyfi,origin,phase-data,', 9041), (',origin,phase-data,shakemap,', 2592), (',earthquake-name,origin,phase-data,', 2172), (',moment-tensor,origin,phase-data,', 1417), (',dyfi,moment-tensor,origin,phase-data,', 1387), (',dyfi,origin,phase-data,shakemap,', 1121), (',internal-moment-tensor,origin,phase-data,', 564), (',internal-moment-tensor,losspager,moment-tensor,origin,phase-data,shakemap,', 560), (',dyfi,moment-tensor,origin,phase-data,shakemap,', 525), (',dyfi,focal-mechanism,nearby-cities,origin,phase-data,scitech-link,', 516), (',losspager,origin,phase-data,shakemap,', 485), (',internal-moment-tensor,moment-tensor,origin,phase-data,', 445), (',dyfi,internal-moment-tensor,moment-tensor,origin,phase-data,', 409), (',dyfi,internal-moment-tensor,losspager,moment-tensor,origin,phase-data,shakemap,', 403), (',dyfi,internal-moment-tensor,internal-origin,losspager,moment-tensor,origin,phase-data,shakemap,', 308), (',dyfi,focal-mecha

In [34]:
##18.Events with high station coverage (nst > threshold).
from tabulate import tabulate
query18 = """select id, place, nst from earthquake_data where nst >50 order by nst desc;"""
cursor_mysql.execute("USE earthquake_db")
cursor_mysql.execute(query18)
result18 = cursor_mysql.fetchall()
headers=["id","place","station_count"]
print(tabulate(result18, headers=headers, tablefmt="fancy_grid"))



╒═══════════════╤═══════════════════════════════════════════════════════════════════╤═════════════════╕
│ id            │ place                                                             │   station_count │
╞═══════════════╪═══════════════════════════════════════════════════════════════════╪═════════════════╡
│ us6000m12f    │ 11 km W of Anamizu, Japan                                         │             619 │
├───────────────┼───────────────────────────────────────────────────────────────────┼─────────────────┤
│ us6000qzfl    │ 120 km ENE of Ozernovskiy, Russia                                 │             566 │
├───────────────┼───────────────────────────────────────────────────────────────────┼─────────────────┤
│ us7000rluk    │ 111 km N of Yakutat, Alaska                                       │             516 │
├───────────────┼───────────────────────────────────────────────────────────────────┼─────────────────┤
│ us7000pvtr    │ Macquarie Island region                       

In [35]:
##Tsunamis & Alerts
##19.Number of tsunamis triggered per year.
from tabulate import tabulate
query19 = """select year(time) as year,count(*) as tsunami_count from earthquake_data where tsunami = 1
group by year(time) order by year;"""
cursor_mysql.execute("USE earthquake_db")
cursor_mysql.execute(query19)
result19 = cursor_mysql.fetchall()
headers = ["year","tsunami_count"]
print(tabulate(result19, headers=headers, tablefmt="psql"))

+--------+-----------------+
|   year |   tsunami_count |
|--------+-----------------|
|   2021 |             114 |
|   2022 |             136 |
|   2023 |             119 |
|   2024 |             114 |
|   2025 |             142 |
|   2026 |              41 |
+--------+-----------------+


In [36]:
##20.Count earthquakes by alert levels (red, orange,etc)
from tabulate import tabulate
query20 = """select alert,count(*) as earthquake_count from earthquake_data where alert is not null group by alert
order by earthquake_count desc;"""
cursor_mysql.execute("USE earthquake_db")
cursor_mysql.execute(query20)
result20 = cursor_mysql.fetchall()
headers = ["alert_level", "earthquake_count"]
print(tabulate(result20, headers=headers, tablefmt="plain"))


alert_level      earthquake_count
green                        4681
yellow                        141
red                            28
orange                         26


In [37]:
##Seismic Pattern & Trends Analysis.
##21.Top 5 countries with the highest average magnitude of earthquakes in the past 5 years       
from tabulate import tabulate
query21 = """select country,avg(mag) as average_magnitude from earthquake_data
where country is not null and mag is not null and time >= date_sub(curdate(),interval 5 year)
group by country order by average_magnitude desc limit 5;"""
cursor_mysql.execute("USE earthquake_db")
cursor_mysql.execute(query21)
result21 = cursor_mysql.fetchall()
headers = ["country", "average_magnitude"]
print(tabulate(result21, headers=headers, tablefmt="psql"))

+-----------------------------------+---------------------+
| country                           |   average_magnitude |
|-----------------------------------+---------------------|
| Russia Earthquake                 |                8.1  |
| Burma (Myanmar) Earthquake        |                7.7  |
| Kahramanmaras earthquake sequence |                7.65 |
| Alaska Earthquake                 |                7.25 |
| Japan Earthquake                  |                7.25 |
+-----------------------------------+---------------------+


In [76]:
##22.countries that have experienced both shallow and deep earthquakes within the same month.
from tabulate import tabulate
query22 = """select country,YEAR(time) as year,MONTH(time) as month from earthquake_data
where country is not null and time is not null and depth_km is not null group by country,YEAR(time),
MONTH(time) having MIN(depth_km) < 50 and MAX(depth_km) >= 50 order by country, year, month;"""
cursor_mysql.execute("USE earthquake_db")
cursor_mysql.execute(query22)
result22 = cursor_mysql.fetchall()
headers = ["country", "year", "month"]
print(tabulate(result22, headers=headers, tablefmt="fancy_grid"))


╒══════════════════════════╤════════╤═════════╕
│ country                  │   year │   month │
╞══════════════════════════╪════════╪═════════╡
│ Afghanistan              │   2021 │       2 │
├──────────────────────────┼────────┼─────────┤
│ Afghanistan              │   2021 │       3 │
├──────────────────────────┼────────┼─────────┤
│ Afghanistan              │   2021 │       4 │
├──────────────────────────┼────────┼─────────┤
│ Afghanistan              │   2021 │       5 │
├──────────────────────────┼────────┼─────────┤
│ Afghanistan              │   2021 │       6 │
├──────────────────────────┼────────┼─────────┤
│ Afghanistan              │   2021 │       7 │
├──────────────────────────┼────────┼─────────┤
│ Afghanistan              │   2021 │       8 │
├──────────────────────────┼────────┼─────────┤
│ Afghanistan              │   2021 │      10 │
├──────────────────────────┼────────┼─────────┤
│ Afghanistan              │   2021 │      11 │
├──────────────────────────┼────────┼───

In [39]:
##23.Compute the year-over-year growth rate in the total number of earthquakes globally.
from tabulate import tabulate
query23=""" select YEAR(time) as year, count(*) as earthquake_count,round((count(*) - lag(count(*)) 
over (order by YEAR(time))) / lag(count(*)) over (order by YEAR(time)) * 100,2) as yoy_growth_rate
from earthquake_data where time is not null group by YEAR(time) order by year;"""
cursor_mysql.execute("USE earthquake_db")
cursor_mysql.execute(query23)
result23 = cursor_mysql.fetchall()
headers = ["year", "earthquake_count", "yoy_growth_rate"]
print(tabulate(result23, headers=headers, tablefmt="fancy_grid"))

╒════════╤════════════════════╤═══════════════════╕
│   year │   earthquake_count │   yoy_growth_rate │
╞════════╪════════════════════╪═══════════════════╡
│   2021 │              21926 │                   │
├────────┼────────────────────┼───────────────────┤
│   2022 │              20208 │             -7.84 │
├────────┼────────────────────┼───────────────────┤
│   2023 │              20830 │              3.08 │
├────────┼────────────────────┼───────────────────┤
│   2024 │              18659 │            -10.42 │
├────────┼────────────────────┼───────────────────┤
│   2025 │              22819 │             22.29 │
├────────┼────────────────────┼───────────────────┤
│   2026 │              14905 │            -34.68 │
╘════════╧════════════════════╧═══════════════════╛


In [43]:
##24.List the 3 most seismically active regions by combining both frequency and average magnitude.
from tabulate import tabulate
query24=""" select country, count(*) as earthquake_count,round(avg(mag), 2) as avg_magnitude,
round(count(*) * avg(mag), 2) as seismic_score from earthquake_data
where country is not null group by country order by seismic_score desc limit 3;"""
cursor_mysql.execute("USE earthquake_db")
cursor_mysql.execute(query24)
result24 = cursor_mysql.fetchall()
headers = ["country","earthquake_count","avg_magnitude","seismic_score"]
print(tabulate(result24, headers=headers, tablefmt="fancy_grid"))

╒═══════════╤════════════════════╤═════════════════╤═════════════════╕
│ country   │   earthquake_count │   avg_magnitude │   seismic_score │
╞═══════════╪════════════════════╪═════════════════╪═════════════════╡
│ Unknown   │              21558 │            4.56 │         98293.3 │
├───────────┼────────────────────┼─────────────────┼─────────────────┤
│ Alaska    │              14841 │            3.47 │         51509.5 │
├───────────┼────────────────────┼─────────────────┼─────────────────┤
│ Indonesia │               9471 │            4.5  │         42631.3 │
╘═══════════╧════════════════════╧═════════════════╧═════════════════╛


In [ ]:
##Depth, Location & Distance-Based  Analysis.
##25. For each country, calculate the average depth of earthquakes within ±5° latitude range of the equator.
from tabulate import tabulate
query25=""" select country, round(avg(depth_km), 2) as average_depth_km from earthquake_data
where country is not null and latitude between -5 and 5 and depth_km  group by country 
order by average_depth_km DESC;"""
cursor_mysql.execute("USE earthquake_db")
cursor_mysql.execute(query25)
result25 = cursor_mysql.fetchall()
headers = ["country","avg_depth"]
print(tabulate(result25, headers=headers, tablefmt="fancy_grid"))

╒══════════════════════════════════╤═════════════╕
│ country                          │   avg_depth │
╞══════════════════════════════════╪═════════════╡
│ Philippines                      │      108.86 │
├──────────────────────────────────┼─────────────┤
│ Papua New Guinea                 │       71.9  │
├──────────────────────────────────┼─────────────┤
│ Peru                             │       63.63 │
├──────────────────────────────────┼─────────────┤
│ Indonesia                        │       62.33 │
├──────────────────────────────────┼─────────────┤
│ Ecuador                          │       61.79 │
├──────────────────────────────────┼─────────────┤
│ Colombia                         │       45.27 │
├──────────────────────────────────┼─────────────┤
│ Unknown                          │       22.93 │
├──────────────────────────────────┼─────────────┤
│ Congo-Uganda                     │       14.77 │
├──────────────────────────────────┼─────────────┤
│ Venezuela                    

In [73]:
##26.Identify countries having the highest ratio of shallow to deep earthquakes.
from tabulate import tabulate
query26="""select country,sum(depth_km < 70) as shallow, sum(depth_km >= 70) as deep,
round(sum(depth_km < 70) / sum(depth_km >= 70), 2) as ratio from earthquake_data
where country is not null group by country having deep > 0 order by ratio desc limit 10;"""
cursor_mysql.execute("USE earthquake_db")
cursor_mysql.execute(query26)
result26 = cursor_mysql.fetchall()
headers = ["country","shallow","deep","ratio"]
print(tabulate(result26, headers=headers, tablefmt="psql"))

+--------------+-----------+--------+---------+
| country      |   shallow |   deep |   ratio |
|--------------+-----------+--------+---------|
| Canada       |       298 |      1 |  298    |
| Iran         |      1004 |      4 |  251    |
| Panama       |       227 |      2 |  113.5  |
| China        |      1476 |     19 |   77.68 |
| Nepal        |       151 |      2 |   75.5  |
| Morocco      |        65 |      1 |   65    |
| Turkey       |       964 |     16 |   60.25 |
| Pakistan     |       246 |      5 |   49.2  |
| India region |       147 |      4 |   36.75 |
| Cyprus       |        36 |      1 |   36    |
+--------------+-----------+--------+---------+


In [64]:
##27.Find the average magnitude difference between earthquakes with tsunami alerts and those without.
from tabulate import tabulate
query27="""select round(avg(case when tsunami = 1 then mag end), 2) as tsunami_avg,
            round(avg(case when tsunami = 0 then mag end), 2) as no_tsunami,
            round(avg(case when tsunami = 1 then mag end) - avg(case when tsunami = 0 then mag end), 2 )
            as mag_dif from earthquake_data;"""
cursor_mysql.execute("USE earthquake_db")
cursor_mysql.execute(query27)
result27 = cursor_mysql.fetchall()
headers =["tsunami_avg","no_tsunami","mag_dif"]
print(tabulate(result27, headers=headers, tablefmt="psql"))

+---------------+--------------+-----------+
|   tsunami_avg |   no_tsunami |   mag_dif |
|---------------+--------------+-----------|
|          5.42 |         4.25 |      1.17 |
+---------------+--------------+-----------+


In [77]:
##28.Using the gap and rms columns, identify events with the lowest data reliability 
##(highest average error margins).
from tabulate import tabulate
query28 = """select id,round(gap, 2) as gap,round(rms, 2) as rms,round((gap + rms) / 2, 2) as err_score
from earthquake_data where gap is not null and rms is not null order by err_score desc limit 10;"""
cursor_mysql.execute("USE earthquake_db")
cursor_mysql.execute(query28)
result28 = cursor_mysql.fetchall()
headers = ["id","gap","rms","err_score"]
print(tabulate(result28, headers=headers, tablefmt="plain"))

id                gap    rms    err_score
pr71519528     359      0.14       179.57
nn00897599     358.18   0.16       179.17
pr2024051000   358      0.16       179.08
pr71508583     356      0.22       178.11
us7000denb     355      0.34       177.67
aka2026fwzivr  354      0.7        177.35
pr2022227000   353      0.61       176.81
pr71489653     352      0.18       176.09
pr2022289000   352      0.07       176.04
pr2021003002   351      0.44       175.72


In [74]:
 ##30.Determine the regions with the highest frequency of deep-focus earthquakes (depth > 300 km).
from tabulate import tabulate
query30 = """select country,count(*) as deep_count from earthquake_data
where depth_km > 300 and country is not null group by country order by deep_count desc limit 10;"""
cursor_mysql.execute("USE earthquake_db")
cursor_mysql.execute(query30)
result30 = cursor_mysql.fetchall()
headers = ["country","deep_count"]
print(tabulate(result30, headers=headers, tablefmt="plain"))

country                     deep_count
Unknown                           3412
Fiji                              1241
Tonga                              628
Indonesia                          296
Japan region                       254
Timor Leste                        223
Wallis and Futuna                  169
Japan                              135
Northern Mariana Islands           133
Philippines                        122
